# 66 — W5 Blind-A submissions: sweep winner + pure-SID

Reads `experiments/cache/w5_sweep/sweep_summary.json` to find the winning weight
from notebook 65. Runs Blind-A on:
  (A) the winning ensemble config (e.g. `170-best-w07-blindsetA.yaml`)
  (B) config 171 (pure-SID)

Produces two CodaBench-ready zips. Wallclock ~3 hr on L4 / ~1.5 hr Blackwell.


In [ ]:
# 1) Setup.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('sid', 'recsys2026_sid_cache'),
    ('dense', 'recsys2026_dense_cache'),
    ('w5_sweep', 'recsys2026_w5_sweep_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

!pip install -q -U "peft>=0.10" "transformers>=4.40" "accelerate>=0.30" "torchao>=0.17"


In [ ]:
# 2) Pick the winning weight from notebook 65's sweep summary, generate Blind-A config.
import json
summary_path = 'experiments/cache/w5_sweep/sweep_summary.json'
assert os.path.exists(summary_path), 'Run notebook 65 first to produce sweep_summary.json'
summary = json.load(open(summary_path))
best_w = summary['best_weight']
print(f'Winner: weight={best_w} (mean_ndcg@20={summary["best_ndcg"]:.4f})')

# Generate Blind-A version: copy base 170 + inject weight (test_dataset_name stays at Blind-A).
from omegaconf import OmegaConf
%cd /content/recsys2026/music-crs-baselines
base = OmegaConf.load('config/170-wrrf-sid-v5kto-blindsetA.yaml')
winner_cfg = OmegaConf.merge(base, OmegaConf.create({'sid_stream_weight': best_w}))
winner_tid = f'170-best-w{int(best_w*10):02d}-blindsetA'
OmegaConf.save(winner_cfg, f'config/{winner_tid}.yaml')
print(f'wrote config/{winner_tid}.yaml')

In [ ]:
# 3) Run Blind-A on the winning ensemble.
!python run_inference_blindset.py \
    --tid {winner_tid} \
    --batch_size 32 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_w5_sweep_cache/blindA_{winner_tid}.log | tail -30

In [ ]:
# 4) Run Blind-A on config 171 (pure-SID).
!python run_inference_blindset.py \
    --tid 171-pure-sid-v5kto-blindsetA \
    --batch_size 32 \
    2>&1 | tee /content/drive/MyDrive/recsys2026_w5_sweep_cache/blindA_171-pure-sid.log | tail -30

In [ ]:
# 5) Validate both prediction files (must be 80 entries each).
%cd /content/recsys2026
for tid in [winner_tid, '171-pure-sid-v5kto-blindsetA']:
    pred_path = f'music-crs-baselines/exp/inference/blindset_A/{tid}.json'
    preds = json.load(open(pred_path))
    n = len(preds) if isinstance(preds, list) else len(preds.keys())
    assert n == 80, f'{tid}: expected 80, got {n}'
    print(f'OK: {tid} has {n} entries')
    !python scripts/validate_prediction.py --input {pred_path}

In [ ]:
# 6) Zip both for CodaBench submission (prediction.json at zip root).
import zipfile, datetime
date_str = datetime.date.today().strftime('%Y-%m-%d')
SUBMIT_DIR = '/content/drive/MyDrive/recsys2026_submissions'
os.makedirs(SUBMIT_DIR, exist_ok=True)
for tid, label in [(winner_tid, f'sid-ensemble-w{int(best_w*10):02d}'),
                   ('171-pure-sid-v5kto-blindsetA', 'sid-pure')]:
    pred_path = f'music-crs-baselines/exp/inference/blindset_A/{tid}.json'
    zip_path = f'{SUBMIT_DIR}/{date_str}-{label}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname='prediction.json')
    print(f'zip ready: {zip_path} ({os.path.getsize(zip_path) / 1024:.1f} KB)')

## After the run

1. Download both zips from `/content/drive/MyDrive/recsys2026_submissions/`.
2. Upload BOTH to CodaBench (note: there may be daily submission quota — sequence them).
3. Record both composite scores in `project_blind_a_w5_results.md`:
   - Winner ensemble (`170-best-w<N>`): composite, nDCG@20, LLM, lex_div
   - Pure-SID (`171`): composite, nDCG@20, LLM, lex_div
4. Compare to:
   - Pre-SID champion (config 132, composite 0.21 if cached)
   - W4 first submission (config 170 weight=0.5)
5. **W5 gate**: Blind-A nDCG@20 >= 0.08 on the winner. If PASS -> freeze for W6/Blind-B. If FAIL -> pivot.

Final choice for W6:
- If ensemble wins on Blind-A composite: ship ensemble.
- If pure-SID wins or ties (per spec §1 promotion path v3): ship pure-SID (simpler pipeline).
